# Evaluation Dataset Creation & Scoring

This notebook creates the evaluation datasets for Model A and Model B,
scores them with DeepEval, and saves the raw scores to JSON.

The analysis notebooks (paired / non-paired) can then load the cached scores
directly without re-running the RAG pipeline or LLM scoring.

### Output files

| File | Description |
|------|-------------|
| `eval_scores.json` | Raw metric scores for both models (paired by item index) |

In [8]:
from bayesAB.resources.preprocessor import Preprocessor

from bayesAB.config import global_config as glob

filename = "Allplan_2020_Manual.pdf"

pre = Preprocessor()

docs = pre.fetch_documents(blob_path=f"{glob.DATA_PKG_DIR}/{filename}", source="local")

documents = pre.chunk_documents(documents=docs)

print(f"Number of processed document chunks: {len(documents)}")

2026-05-04 00:23:01,974 - ai_eval.resources.preprocessor - INFO - Processed /Users/avosseler/Github/Team/tum-hackathon/data/Allplan_2020_Manual.pdf
2026-05-04 00:23:01,983 - ai_eval.resources.preprocessor - INFO - Using SentenceChunker for document chunking.


Finished 'fetch_documents' in 1.9783 secs


🦛 choooooooooooooooooooonk 100% • 319/319 docs chunked [00:02<00:00, 139.44doc/s] 🌱
2026-05-04 00:23:04,660 - ai_eval.resources.preprocessor - INFO - Chunked 319 pages into 319 chunks.


Finished 'chunk_documents' in 2.6873 secs
Number of processed document chunks: 319


## Load annotated QA data

In [9]:
from bayesAB.config import global_config as glob
from bayesAB.services.file import JSONService

json_svc = JSONService(path="generated_qa_data_tum.json", root_path=glob.DATA_PKG_DIR, verbose=True)

qa_data = json_svc.doRead()
print(f"Number of evaluation data samples: {len(qa_data)}")

2026-05-04 00:23:11,475 - ai_eval.services.file - INFO - Read: /Users/avosseler/Github/Team/tum-hackathon/data/generated_qa_data_tum.json


Number of evaluation data samples: 100


In [10]:
ground_truth_contexts = [item["context"] for item in qa_data]
sample_queries = [item["question"] for item in qa_data]
expected_responses = [item["answer"] for item in qa_data]

print(f"Prepared {len(sample_queries)} queries for evaluation")

Prepared 100 queries for evaluation


## 3. Set up RAG models (A and B)

In [11]:
from bayesAB.resources.rag_template import TFIDFRAG
from langchain_google_vertexai import ChatVertexAI

from bayesAB.config import global_config as glob

# Two generators, same retrieval
generator_A = ChatVertexAI(
    project=glob.GCP_PROJECT,
    model_name="gemini-2.5-flash",
    temperature=0.1,
    max_retries=2,
)

generator_B = ChatVertexAI(
    project=glob.GCP_PROJECT,
    model_name="gemini-2.0-flash",
    temperature=0.1,
    max_retries=2,
)

# Same retriever (TFIDF, k=5) for both — only the generator differs
rag_A = TFIDFRAG(generator_A, documents, k=5)
rag_B = TFIDFRAG(generator_B, documents, k=5)

print("Model A: gemini-2.5-flash")
print("Model B: gemini-2.0-flash")
print("Retriever: TFIDF (k=5) — shared")

Model A: gemini-2.5-flash
Model B: gemini-2.0-flash
Retriever: TFIDF (k=5) — shared


## 4. Build evaluation datasets and score with DeepEval

In [12]:
from bayesAB.resources import deepeval_scorer as deep
from bayesAB.resources import eval_dataset_builder as eval_builder

# Build evaluation dataset for Model A (gemini-2.5-flash)
builder_A = eval_builder.EvalDatasetBuilder(rag_A)

eval_dataset_A = builder_A.build_evaluation_dataset(
    input_contexts=ground_truth_contexts,
    sample_queries=sample_queries,
    expected_responses=expected_responses,
)

scorer_A = deep.DeepEvalScorer(eval_dataset_A)

results_A = scorer_A.calculate_scores()
summary_A = scorer_A.get_summary(save_to_file=False)

print(f"Model A (gemini-2.5-flash): {len(summary_A)} test cases scored")

Building test dataset:   0%|          | 0/100 [00:00<?, ?it/s]

2026-05-04 00:28:14,961 - ai_eval.resources.eval_dataset_builder - INFO - Evaluation dataset created successfully.
/Users/avosseler/Github/Team/tum-hackathon/.venv/lib/python3.12/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
2026-05-04 00:28:16,675 - ai_eval.resources.get_models - INFO - Using chat model: gemini-2.5-flash
2026-05-04 00:28:16,678 - ai_eval.resources.get_models - INFO - Using judge model: gemini-2.5-flash
2026-05-04 00:28:16,678 - ai_eval.resources.get_models - INFO - Using embedding model: text-multilingual-embedding-002
2026-05-04 00:28:16,678 - ai_eval.resources.get_models - INFO - Models loaded successfully
2026-05-04 00:28:16,680 - ai_eval.resources.deepeval_scorer - INFO - Starting evaluation...


✨ You're running DeepEval's latest Answer Relevancy Metric! (using LangChain Model, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using LangChain Model, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using LangChain Model, strict=False, 
async_mode=True)...

...

Evaluating 100 test case(s) in parallel: |          |  0% (0/100) [Time Taken: 00:00, ?test case/s]/Users/avosseler/Github/Team/tum-hackathon/src/ai_eval/utils/utils.py:157: LangChainDeprecationWarning: The method `BaseChatModel.apredict` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~ainvoke` instead.
  res = await chat_model.apredict(prompt)
I0000 00:00:1777847303.830402 5537594 ssl_transport_security.cc:1884] Handshake failed with error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate
I0000 00:00:1777847312.265256 5537593 ssl_transport_security.cc:1884] Handshake failed with error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate
Retrying langchain_google_vertexai.chat_models._acompletion_with_retry.<locals>._completion_with_retry_inner in 4.0 seconds as it raised ResourceExhausted: 429 Resource exhau



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: LangChain Model, reason: The score is 1.00 because the output is perfectly relevant to the input query, addressing exactly how to sort documents without any extraneous information. Excellent work!, error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: LangChain Model, reason: The score is 1.00 because there are no contradictions, indicating excellent faithfulness to the retrieval context. Great job!, error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.7, strict: False, evaluation model: LangChain Model, reason: The score is 1.00 because the expected output is perfectly supported by the 1st node in the retrieval context!, error: None)
  - ✅ Contextual Precision (score: 1.0, threshold: 0.7, strict: False, evaluation model: LangChain Model, reason: The score is 1.00 because all relevant nodes are perfectly ranked at the top!, error: Non

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

2026-05-04 00:30:37,791 - ai_eval.resources.deepeval_scorer - INFO - Evaluation completed successfully!
2026-05-04 00:30:37,792 - ai_eval.resources.deepeval_scorer - INFO - Summary of test results generated successfully!


Model A (gemini-2.5-flash): 100 test cases scored


In [13]:
# Build evaluation dataset for Model B (gemini-2.0-flash)
builder_B = eval_builder.EvalDatasetBuilder(rag_B)

eval_dataset_B = builder_B.build_evaluation_dataset(
    input_contexts=ground_truth_contexts,
    sample_queries=sample_queries,
    expected_responses=expected_responses,
)

scorer_B = deep.DeepEvalScorer(eval_dataset_B)
results_B = scorer_B.calculate_scores()
summary_B = scorer_B.get_summary(save_to_file=False)

print(f"Model B (gemini-2.0-flash): {len(summary_B)} test cases scored")

Building test dataset:   0%|          | 0/100 [00:00<?, ?it/s]

I0000 00:00:1777847437.844702 5505730 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1777847452.344250 5541794 ssl_transport_security.cc:1884] Handshake failed with error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate
2026-05-04 00:32:00,580 - ai_eval.resources.eval_dataset_builder - INFO - Evaluation dataset created successfully.
/Users/avosseler/Github/Team/tum-hackathon/.venv/lib/python3.12/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
2026-05-04 00:32:01,540 - ai_eval.resources.get_models - INFO - Using chat model: gemini-2.5-flash
2026-05-04 00:32:01,541 - ai_eval.resources.get_models - INF

✨ You're running DeepEval's latest Answer Relevancy Metric! (using LangChain Model, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using LangChain Model, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using LangChain Model, strict=False, 
async_mode=True)...

...

Evaluating 100 test case(s) in parallel: |          |  0% (0/100) [Time Taken: 00:00, ?test case/s]I0000 00:00:1777847527.393762 5541797 ssl_transport_security.cc:1884] Handshake failed with error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate
Evaluating 100 test case(s) in parallel: |██████████|100% (100/100) [Time Taken: 02:13,  1.34s/test case]



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: LangChain Model, reason: The score is 1.00 because the output is perfectly relevant to the input, addressing all aspects of the query without any irrelevant information. Great job!, error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: LangChain Model, reason: The score is 1.00 because there are no contradictions, indicating excellent faithfulness to the retrieval context. Great job!, error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.7, strict: False, evaluation model: LangChain Model, reason: The score is 1.00 because the entire expected output is perfectly supported by the 1st node in the retrieval context. Excellent work!, error: None)
  - ✅ Contextual Precision (score: 1.0, threshold: 0.7, strict: False, evaluation model: LangChain Model, reason: The score is 1.00 because all relevant information was perfectly prioritized and p

✓ Tests finished 🎉! Run 'deepeval login' to save and analyze evaluation results on Confident AI.
 
✨👀 Looking for a place for your LLM test data to live 🏡❤️ ? Use Confident AI to get & share testing reports, 
experiment with models/prompts, and catch regressions for your LLM system. Just run 'deepeval login' in the CLI.

2026-05-04 00:34:15,346 - ai_eval.resources.deepeval_scorer - INFO - Evaluation completed successfully!
2026-05-04 00:34:15,347 - ai_eval.resources.deepeval_scorer - INFO - Summary of test results generated successfully!


Model B (gemini-2.0-flash): 100 test cases scored


## 5. Save raw scores to JSON

We save **raw continuous scores** per metric. Each analysis notebook can then:
- Load the scores with `JSONService`
- Apply its own binarization threshold
- Use as paired (same item index) or non-paired (treat groups as independent)

In [16]:
from datetime import datetime

import numpy as np

from bayesAB.config import global_config as glob
from bayesAB.services.file import JSONService

metric_names = ["Answer Relevancy", "Faithfulness"]

scores_to_save = {
    "timestamp": datetime.now().isoformat(),
    "model_A": "gemini-2.5-flash",
    "model_B": "gemini-2.0-flash",
    "retriever": "TFIDF (k=5)",
    "n_items": None,
    "metrics": {},
}

for metric in metric_names:
    scores_A = np.array(
        [
            test_data["metrics"][metric]
            for test_data in summary_A.values()
            if test_data["metrics"].get(metric) is not None
        ]
    )
    scores_B = np.array(
        [
            test_data["metrics"][metric]
            for test_data in summary_B.values()
            if test_data["metrics"].get(metric) is not None
        ]
    )

    n = min(len(scores_A), len(scores_B))
    scores_to_save["metrics"][metric] = {
        "s_A_raw": scores_A[:n].tolist(),
        "s_B_raw": scores_B[:n].tolist(),
    }

scores_to_save["n_items"] = n

# Write to JSON
scores_svc = JSONService(path="eval_scores.json", root_path=glob.DATA_PKG_DIR)
scores_svc.doWrite(scores_to_save)

print(f"Scores saved to {scores_svc.path}")
print(f"  Models: {scores_to_save['model_A']} vs {scores_to_save['model_B']}")
print(f"  Items: {n}")
print(f"  Metrics: {metric_names}")
for metric in metric_names:
    m = scores_to_save["metrics"][metric]
    s_A = np.array(m["s_A_raw"])
    s_B = np.array(m["s_B_raw"])
    print(f"  {metric}: mean_A={s_A.mean():.3f}, mean_B={s_B.mean():.3f}")

2026-05-04 00:34:40,833 - ai_eval.services.file - INFO - JSON Service Output to File: /Users/avosseler/Github/Team/tum-hackathon/data/eval_scores.json


Scores saved to /Users/avosseler/Github/Team/tum-hackathon/data/eval_scores.json
  Models: gemini-2.5-flash vs gemini-2.0-flash
  Items: 100
  Metrics: ['Answer Relevancy', 'Faithfulness']
  Answer Relevancy: mean_A=0.975, mean_B=0.980
  Faithfulness: mean_A=0.985, mean_B=0.959


## 6. Verify: reload and inspect

Quick sanity check — this is the same pattern the analysis notebooks will use.

In [17]:
import numpy as np

from bayesAB.config import global_config as glob
from bayesAB.services.file import JSONService

scores_svc = JSONService(path="eval_scores.json", root_path=glob.DATA_PKG_DIR, verbose=True)
cached = scores_svc.doRead()

print(f"Loaded scores from {scores_svc.path}")
print(f"  Models: {cached['model_A']} vs {cached['model_B']}")
print(f"  Timestamp: {cached['timestamp']}")
print(f"  Items: {cached['n_items']}")
print(f"  Metrics: {list(cached['metrics'].keys())}")

for metric, m in cached["metrics"].items():
    s_A = np.array(m["s_A_raw"])
    s_B = np.array(m["s_B_raw"])
    print(f"\n  {metric}:")
    print(f"    n_A={len(s_A)}, n_B={len(s_B)}")
    print(f"    mean_A={s_A.mean():.3f}, mean_B={s_B.mean():.3f}")
    print(f"    range_A=[{s_A.min():.3f}, {s_A.max():.3f}]")
    print(f"    range_B=[{s_B.min():.3f}, {s_B.max():.3f}]")

2026-05-04 00:34:42,860 - ai_eval.services.file - INFO - Read: /Users/avosseler/Github/Team/tum-hackathon/data/eval_scores.json


Loaded scores from /Users/avosseler/Github/Team/tum-hackathon/data/eval_scores.json
  Models: gemini-2.5-flash vs gemini-2.0-flash
  Timestamp: 2026-05-04T00:34:40.832923
  Items: 100
  Metrics: ['Answer Relevancy', 'Faithfulness']

  Answer Relevancy:
    n_A=100, n_B=100
    mean_A=0.975, mean_B=0.980
    range_A=[0.400, 1.000]
    range_B=[0.000, 1.000]

  Faithfulness:
    n_A=100, n_B=100
    mean_A=0.985, mean_B=0.959
    range_A=[0.500, 1.000]
    range_B=[0.000, 1.000]


I0000 00:00:1777847802.604385 5541797 ssl_transport_security.cc:1884] Handshake failed with error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate
I0000 00:00:1777847942.696455 5541795 ssl_transport_security.cc:1884] Handshake failed with error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate
I0000 00:00:1777848087.805164 5541793 ssl_transport_security.cc:1884] Handshake failed with error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate
I0000 00:00:1777848187.886035 5541792 ssl_transport_security.cc:1884] Handshake failed with error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate
I0000 00:00:1777848312.972154 5541803 ssl_transport_security.cc:1884] Handshake failed with error SSL_ER